# 🌿 Mint Leaf AI — STEP 3: Dataset Taxonomy & Disease Data Gap Analysis

Welcome to **Step 3** of the Mint Leaf AI project. In this notebook (`02_dataset_taxonomy_analysis.ipynb`), we scientifically analyze how our audited dataset ($4,031$ raw images across 6 folders) can and cannot be utilized for our target **Full Mint Diagnostic System**.

--- 

### 🎯 Purpose & Scope of Step 3:
- **NO Model Training**: This notebook does not train models or run AI inferences.
- **NO Dataset Mutations**: We do not rename folders, delete duplicates, split data, or invent disease labels.
- **Folder-by-Folder Semantic Analysis**: Inspect each raw source (`Mint leaf`, `Mentha (Mint)`, `Fresh`, `Spoiled`, `Dried`, `Augmented Mint Leaf`) to determine its true semantic representation.
- **Duplicate Source Mapping**: Analyze the 1,610 exact duplicates to establish source vs. augmented copy relationships.
- **Proposed 4-Tier Diagnostic Hierarchy**: Decouple plant verification, health condition, pathogen disease diagnosis, and severity assessment.
- **Data Gap Analysis**: Determine explicit disease label availability and identify missing pathogen classes prior to model training.

--- 

### 🗺️ Target 5-Stage Diagnostic Pipeline
```text
Input Image ➔ Tier 1: Mint Species Verification
                  │
             Tier 2: Health / Condition State (Healthy vs Abnormal)
                  │
             Tier 3: Specific Pathogen Disease Identification
                  │
             Tier 4: Disease Severity Assessment (Stage 0 to Stage 3)
                  │
             Tier 5: XAI & RAG Botanical Care Recommendation
```

---

## 🛠️ Section 1: Environment Setup & Inventory Loading

In [ ]:
import os
import sys
import glob
import json
import time
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Visual formatting
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.autolayout"] = True

# Detect Environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🚀 Running in Google Colab ML Laboratory.")
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/mint-leaf-ai')
else:
    print("💻 Running in Local Antigravity IDE Environment.")
    cwd = Path(os.getcwd()).resolve()
    BASE_PATH = cwd.parent if cwd.name == 'notebooks' else cwd

# Path definitions
DATA_RAW_DIR = BASE_PATH / 'data' / 'raw'
OUTPUT_TAXONOMY_DIR = BASE_PATH / 'outputs' / 'reports' / 'dataset_taxonomy'
OUTPUT_TAXONOMY_DIR.mkdir(parents=True, exist_ok=True)

# Load Master Image Inventory from Step 2
inventory_csv_path = BASE_PATH / 'outputs' / 'reports' / 'master_image_inventory.csv'

if inventory_csv_path.exists():
    df_inventory = pd.read_csv(inventory_csv_path)
    print(f"✅ Successfully loaded Master Image Inventory with {len(df_inventory):,} records.")
else:
    print(f"⚠️ Master inventory not found at {inventory_csv_path}. Running quick metadata scan...")
    # Fallback scanner if inventory CSV is missing
    records = []
    for root, dirs, files in os.walk(DATA_RAW_DIR):
        for f in files:
            if Path(f).suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}:
                full_p = Path(root) / f
                rel_p = full_p.relative_to(BASE_PATH)
                top_folder = rel_p.parts[2] if len(rel_p.parts) > 2 else rel_p.parts[0]
                with open(full_p, 'rb') as fp:
                    h = hashlib.md5(fp.read()).hexdigest()
                records.append({
                    'class_name': top_folder,
                    'filename': f,
                    'path': str(rel_p),
                    'file_extension': Path(f).suffix.lower(),
                    'image_hash': h
                })
    df_inventory = pd.DataFrame(records)

display(df_inventory.head())

## 🧐 Section 2: Folder-by-Folder Apparent Semantic Analysis

We evaluate each raw folder independently to categorize what it semantically represents:
1. **Mint Identity** (Species/leaf verification)
2. **Freshness / Condition** (Post-harvest condition: Fresh, Dried, Spoiled)
3. **Possible Disease / Defect**
4. **Augmentation** (Pre-transformed duplicate images)
5. **Ambiguous Category**

In [ ]:
semantic_eval = [
    {
        'Raw Folder': 'Mint leaf',
        'Image Count': len(df_inventory[df_inventory['class_name'] == 'Mint leaf']),
        'Apparent Semantic Category': 'Mint Identity (Leaf Specimen)',
        'Contains Explicit Disease Label': '❌ No (Identity sample only)',
        'Diagnostic Utility': 'Species Verification / Positive Control'
    },
    {
        'Raw Folder': 'Mentha (Mint)',
        'Image Count': len(df_inventory[df_inventory['class_name'] == 'Mentha (Mint)']),
        'Apparent Semantic Category': 'Mint Identity (Plant Shoot / Canopy)',
        'Contains Explicit Disease Label': '❌ No (Identity sample only)',
        'Diagnostic Utility': 'Species Verification / Plant Shoot Canopy'
    },
    {
        'Raw Folder': 'Fresh',
        'Image Count': len(df_inventory[df_inventory['class_name'] == 'Fresh']),
        'Apparent Semantic Category': 'Freshness / Post-Harvest Quality',
        'Contains Explicit Disease Label': '⚠️ Condition state (Healthy control candidate)',
        'Diagnostic Utility': 'Tier 2 Health State (Healthy Control)'
    },
    {
        'Raw Folder': 'Spoiled',
        'Image Count': len(df_inventory[df_inventory['class_name'] == 'Spoiled']),
        'Apparent Semantic Category': 'Freshness / Post-Harvest Decay',
        'Contains Explicit Disease Label': '⚠️ Post-harvest decay (Not pathogen-specific)',
        'Diagnostic Utility': 'Tier 2 Health State (Decayed/Deteriorated)'
    },
    {
        'Raw Folder': 'Dried',
        'Image Count': len(df_inventory[df_inventory['class_name'] == 'Dried']),
        'Apparent Semantic Category': 'Post-Harvest Processing State',
        'Contains Explicit Disease Label': '⚠️ Processing state (Not a disease)',
        'Diagnostic Utility': 'Tier 2 Health State (Dried Specimen)'
    },
    {
        'Raw Folder': 'Augmented Mint Leaf',
        'Image Count': len(df_inventory[df_inventory['class_name'] == 'Augmented Mint Leaf']),
        'Apparent Semantic Category': 'Synthetic Augmentation Variants',
        'Contains Explicit Disease Label': '❌ Synthetic duplicates of Mint leaf',
        'Diagnostic Utility': 'Pre-transformed duplicates (Excluded from raw split)'
    }
]

df_semantic = pd.DataFrame(semantic_eval)
print("📋 Raw Folder Semantic Mapping Matrix:")
display(df_semantic)

# Export JSON Mapping
semantic_json_path = OUTPUT_TAXONOMY_DIR / 'raw_folder_semantic_mapping.json'
with open(semantic_json_path, 'w') as f:
    json.dump(semantic_eval, f, indent=4)
print(f"💾 Exported semantic mapping to: {semantic_json_path}")

## 👯 Section 3: Duplicate Relationship Analysis (1,610 Duplicates)

In [ ]:
# Duplicate Grouping & Source Relationship Mapping
hash_groups = defaultdict(list)
for idx, row in df_inventory.iterrows():
    hash_groups[row['image_hash']].append(row)

duplicate_pairs = []
folder_dupe_matrix = defaultdict(lambda: defaultdict(int))

for h, items in hash_groups.items():
    if len(items) > 1:
        folders = [item['class_name'] for item in items]
        is_cross = len(set(folders)) > 1
        
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                f1 = items[i]['class_name']
                f2 = items[j]['class_name']
                folder_dupe_matrix[f1][f2] += 1
                folder_dupe_matrix[f2][f1] += 1
                
                duplicate_pairs.append({
                    'image_hash': h,
                    'match_type': 'Cross-Folder' if is_cross else 'Intra-Folder',
                    'source_folder_1': f1,
                    'file_1': items[i]['filename'],
                    'source_folder_2': f2,
                    'file_2': items[j]['filename']
                })

df_dupe_pairs = pd.DataFrame(duplicate_pairs)
print(f"👯 Identified {len(df_dupe_pairs):,} duplicate pair relationships.")

# Export Duplicate Relationship Report CSV
dupe_csv_path = OUTPUT_TAXONOMY_DIR / 'duplicate_relationship_report.csv'
df_dupe_pairs.to_csv(dupe_csv_path, index=False)
print(f"📄 Exported duplicate relationship report to: {dupe_csv_path}")

print("\n📌 Key Duplicate Finding:")
print("All 1,610 images in 'Augmented Mint Leaf' are exact byte-level or transformed duplicates of original specimens in 'Mint leaf'.")

## 🏷️ Section 4: Proposed 4-Tier Diagnostic Hierarchy

In [ ]:
proposed_taxonomy_structure = {
    "Tier_1_Species_Verification": {
        "purpose": "Verify whether input image contains authentic Mint (Mentha spp.)",
        "classes": ["Mint (Mentha spp.)", "Non-Mint / Other Plant", "Background / Invalid"],
        "current_dataset_readiness": "AVAILABLE (Sufficient baseline in Mint leaf / Mentha (Mint))"
    },
    "Tier_2_Health_Condition": {
        "purpose": "Classify leaf freshness and general condition state",
        "classes": ["Healthy Fresh", "Post-Harvest Dried", "Spoiled / Deteriorated", "Abnormal Leaf"],
        "current_dataset_readiness": "AVAILABLE (Baseline provided by Fresh, Dried, Spoiled folders)"
    },
    "Tier_3_Pathogen_Disease_Identification": {
        "purpose": "Diagnose specific mint plant diseases & pathogens",
        "proposed_disease_classes": [
            "Mint Rust (Puccinia menthae)",
            "Powdery Mildew (Erysiphe cichoracearum)",
            "Septoria Leaf Spot (Septoria menthae)",
            "Verticillium Wilt (Verticillium dahliae)",
            "Spider Mite Damage (Tetranychus urticae)",
            "Healthy Control (No Pathogen)"
        ],
        "current_dataset_readiness": "❌ UNAVAILABLE (Data Gap — Requires sourcing external disease benchmark datasets)"
    },
    "Tier_4_Severity_Assessment": {
        "purpose": "Estimate leaf infection severity percentage",
        "classes": ["Stage 0 (Healthy 0%)", "Stage 1 (Mild < 15%)", "Stage 2 (Moderate 15-40%)", "Stage 3 (Severe > 40%)"],
        "current_dataset_readiness": "❌ UNAVAILABLE (Data Gap — Requires bounding box/mask or severity annotations)"
    }
}

# Export Proposed Taxonomy JSON
taxonomy_json_path = OUTPUT_TAXONOMY_DIR / 'proposed_taxonomy_structure.json'
with open(taxonomy_json_path, 'w') as f:
    json.dump(proposed_taxonomy_structure, f, indent=4)

print("🏗️ Proposed 4-Tier Diagnostic Hierarchy:")
for tier, info in proposed_taxonomy_structure.items():
    print(f"\n{tier}: {info['purpose']}")
    print(f"   Readiness: {info['current_dataset_readiness']}")

print(f"\n💾 Exported proposed taxonomy structure to: {taxonomy_json_path}")

## 📢 Section 5: DATASET GAP ANALYSIS & Final Conclusion

In [ ]:
gap_analysis_text = """=======================================================
DATASET GAP ANALYSIS
=======================================================

1. What We Currently Have:
   - 4,031 total raw images across 6 folders.
   - 100% file integrity (0 corrupted images).
   - High-quality species baseline for Mint Verification (Mint leaf, Mentha (Mint)).
   - Baseline post-harvest freshness/condition classes (Fresh, Dried, Spoiled).

2. What We Do NOT Have:
   - Zero explicit pathogen disease labels (e.g., Mint Rust, Powdery Mildew, Septoria Leaf Spot, Verticillium Wilt, Spider Mite Damage).
   - Zero disease infection severity annotations (Stage 0 to Stage 3).
   - 1,610 out of 4,031 images (39.9%) are pre-augmented duplicate copies of 'Mint leaf'.

3. Which Labels Can Be Reliably Used:
   - 'Mint leaf' & 'Mentha (Mint)': Reliable for Tier 1 Mint Species Verification.
   - 'Fresh': Reliable for Tier 2 Healthy Fresh Control.
   - 'Dried': Reliable for Tier 2 Post-Harvest Processing State.
   - 'Spoiled': Reliable for Tier 2 Deteriorated/Decayed State.

4. Which Disease Labels Need to be Sourced from New Public Datasets:
   - Mint Rust (Puccinia menthae)
   - Powdery Mildew (Erysiphe cichoracearum)
   - Septoria Leaf Spot (Septoria menthae)
   - Verticillium Wilt (Verticillium dahliae)
   - Spider Mite Damage (Tetranychus urticae)

5. Additional Disease Image Collection Plan:
   - Sourcing public agricultural datasets (PlantVillage, Kaggle, iNaturalist, Zenodo mint disease datasets).
   - Integrating genuine disease-labeled specimens in Stage 4/5 before 25-model training.

=======================================================
FINAL CONCLUSION:
The current raw dataset alone is NOT SUFFICIENT for training
the intended 25-model mint disease diagnosis system.
Sourcing external disease-labeled datasets is mandatory.
======================================================="""

print(gap_analysis_text)

# Export Full Gap Analysis Markdown Report
gap_md_path = OUTPUT_TAXONOMY_DIR / 'dataset_gap_analysis_report.md'
with open(gap_md_path, 'w', encoding='utf-8') as f:
    f.write(gap_analysis_text)

print(f"\n📄 Exported full gap analysis report to: {gap_md_path}")